# RUNG-26b SCORE (jax only) — AF2 scores the RFdiffusion binder sequences

Clean jax/ColabDesign env (NO torch — torch+jax cuDNN collide). The RUNG-26-proven AF2 scoring config. Loads
the binder sequences the DESIGN notebook saved + the folded target, scores each vs MUT & WT pMHC, ranks.
Run AFTER the design notebook produced sequences. Cell1 install -> 2 score+rank.

In [ ]:
#@title Cell 1 — clone + GPU + install ColabDesign (jax; NO torch)  [~5 min]
import os, glob, subprocess
from pathlib import Path
REPO=Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!nvidia-smi -L
from google.colab import drive; drive.mount('/content/drive')
def sh(c): r=subprocess.run(c,shell=True,capture_output=True,text=True); (r.returncode and print('  !',(r.stderr or '')[-300:])); return r.returncode
sh('pip install -q gemmi git+https://github.com/sokrypton/ColabDesign.git@v1.1.1')
if not glob.glob('params/params_model_*.npz'):
    sh('apt-get -qq install -y aria2 >/dev/null; mkdir -p params && cd params && (aria2c -q -x16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar || wget -q https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar) && tar -xf alphafold_params_2022-12-06.tar && rm -f alphafold_params_2022-12-06.tar')
from colabdesign import mk_afdesign_model; print('colabdesign import OK')
print('[CELL 1] done')

In [ ]:
#@title Cell 2 — score every binder seq vs MUT+WT, rank  [time-boxed, resumable]
max_hours=3.0 #@param {type:'number'}
import os, glob, json, time
from google.colab import drive; drive.mount('/content/drive')
M=json.load(open('/content/drive/MyDrive/cancer-recon/rung26b_rfdiff/meta.json'))
MUT_PDB=M['mut_pdb']; WT_PDB=M['wt_pdb']; HOT=f"B{M['hotspot']}"; DES=f"{M['work']}/designs"
from colabdesign import mk_afdesign_model, clear_mem
def score(pdb,seq):
    clear_mem(); m=mk_afdesign_model(protocol='binder',use_multimer=False,num_recycles=3,data_dir='.')
    m.prep_inputs(pdb_filename=pdb,chain='A,B',binder_len=len(seq),hotspot=HOT,rm_target_seq=False,ignore_missing=True)
    m.predict(seq=seq,verbose=False); log=m.aux['log']
    return {'pae_interaction':round(float(log['i_pae'])*31.0,3),'binder_plddt':round(float(log['plddt'])*100.0,1)}
seqs=sorted(glob.glob(f'{DES}/*/seq.json')); t_end=time.time()+max_hours*3600
print(f'[CELL 2] {len(seqs)} sequences to score')
for sf in seqs:
    if time.time()>t_end: print('timebox hit'); break
    dd=os.path.dirname(sf); mj=f'{dd}/metrics.json'
    if os.path.exists(mj): continue
    j=json.load(open(sf))
    if 'sequence' not in j: continue
    try:
        seq=j['sequence']; mut=score(MUT_PDB,seq); rec={**j,'mut':mut}
        if mut['pae_interaction']<=10.0 and mut['binder_plddt']>=80.0: rec['wt']=score(WT_PDB,seq)
        json.dump(rec, open(mj,'w'))
        print(f"  {j['design_id']}: mut_pae={mut['pae_interaction']} plddt={mut['binder_plddt']}"+(f" DISC={round(rec['wt']['pae_interaction']-mut['pae_interaction'],2)}" if 'wt' in rec else ' (not a binder)'), flush=True)
    except Exception as e: print(f"  {j['design_id']} score FAIL: {e}", flush=True)
# rank + bundle
import sys, zipfile, shutil
dst='runs/rung26b_rfdiff'; os.makedirs(dst,exist_ok=True)
for m in glob.glob(f'{DES}/*/metrics.json'):
    d=os.path.join(dst,os.path.basename(os.path.dirname(m))); os.makedirs(d,exist_ok=True); shutil.copy(m,d)
os.system(f'{sys.executable} scripts/52_binder_design.py rank {dst}')
b='/content/rung26b_rfdiff.zip'
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in glob.glob(f'{dst}/**/*',recursive=True):
        if os.path.isfile(x): z.write(x,arcname=os.path.relpath(x,'runs'))
print('[CELL 2] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(skip dl:',e,')')